In [28]:

import os
import openai
from openai import OpenAI
import pandas as pd

# Diese beiden Codes sind individuell und müssen selbst auf der Website der KI Toolbox generiert werden.
# Leider steht diese Anwendung vorerst nur für Mitarbeiter:innen zur Verfügung.

try:
    api_key = os.environ["OPENAI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Environment variable OPENAI_API_KEY missing.") from exc

In [29]:
MODEL = 'gpt-5-nano'

In [34]:
client = OpenAI(api_key=api_key)

def send_LLM(prompt: str) -> str:
    try:
        # Send chat request
        chat_completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a rigorous JSON-only annotator of online sarcasm."},
                {"role": "user", "content": prompt}
            ])
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return chat_completion.choices[0].message.content

In [31]:
import json

SARCASM_SCHEMA = {
    "sarcasm": {
        "sarcasm_score": "1-5 integer (1 = not sarcastic, 5 = very sarcastic)",
        "confidence_score": "1-5 integer (1 = not confident, 5 = very confident)",
        "justification": "string - explanation based on observed language",
        "sarcasm_markers": ["string"]
    } 
}

def prompt_sarcasm_analysis(thread_title: str, comment: str) -> str:
    schema_str = json.dumps(SARCASM_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online sarcasm.
Analyze the COMMENT below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

THREAD TITLE:
\"\"\"{thread_title}\"\"\"

COMMENT:
\"\"\"{comment}\"\"\"

"""

In [42]:
df = pd.read_csv('data.csv')

df.head()

,id,thread_title,comment
0,1,Mehrheit sieht ältere im Vorteil – nicht einma...,Solange alle ihre Tiefkühlpizza und RTL haben ...
1,2,Mehrheit sieht ältere im Vorteil – nicht einma...,Ich bin schockiert.
2,3,Mehrheit sieht ältere im Vorteil – nicht einma...,"Würden sich die babyboomer zu Tode arbeiten, b..."
3,4,Mehrheit sieht ältere im Vorteil – nicht einma...,Auf den Schreck erstmal eine Rentenerhöhung.
4,5,Mehrheit sieht ältere im Vorteil – nicht einma...,Finde den Generationen-Vertrag an sich schon d...


In [43]:
import json

results = []

for _, row in df.iterrows():
    row_id = row["id"]
    thread_title = row["thread_title"]
    comment = row["comment"]

    prompt = prompt_sarcasm_analysis(thread_title, comment)
    llm_answer = send_LLM(prompt)

    try:
        parsed = json.loads(llm_answer)
        sarcasm_obj = parsed.get("sarcasm", {})
    except json.JSONDecodeError:
        sarcasm_obj = {"raw_response": llm_answer}

    results.append({
        "id": row_id,
        "thread_title": thread_title,
        "comment": comment,
        "sarcasm": sarcasm_obj
    })

# Save concatenated results as JSON array
with open("sarcasm_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Optional: show in notebook
results[:2]

[{'id': 1,
  'thread_title': 'Mehrheit sieht ältere im Vorteil – nicht einmal ein Drittel der jungen glaubt an faire Rente',
  'comment': 'Solange alle ihre Tiefkühlpizza und RTL haben ist doch alles super!',
  'sarcasm': {'sarcasm_score': 4,
   'confidence_score': 4,
   'justification': "The comment uses a superficially positive claim 'alles super' to mock the situation concerning pension fairness, with 'Solange ... ist doch alles super!' signaling irony. The reference to trivial comforts (frozen pizza, RTL) serves to belittle the broader issue, indicating sarcasm.",
   'sarcasm_markers': ['irony / sarcasm through opposite of stated sentiment',
    "use of 'doch' for emphasis",
    'minimization / trivialization of a serious issue',
    'mocking tone directed at reader/viewer context']}},
 {'id': 2,
  'thread_title': 'Mehrheit sieht ältere im Vorteil – nicht einmal ein Drittel der jungen glaubt an faire Rente',
  'comment': 'Ich bin schockiert.',
  'sarcasm': {'sarcasm_score': 1,
   '